# Two packages, one learner lifecycle
`experiment` owns protocols; `models` owns learning state. This notebook calls real package implementations rather than maintaining its own training/evaluation code. All data here are synthetic software fixtures, **not Amazon results**.

Read `docs/PACKAGES-AND-BUILDING.md` for the ABC and runnable model-development examples. The separate minimal reference is tagged `boilerplate-v1`. Use this project's uv interpreter in your existing Jupyter installation.

## SASRec: existing cutoff-pilot lifecycle
The CLI demo generates local synthetic inputs, prepares isolated holdouts, selects initial weights on validation NDCG, verifies checkpoint reload, and performs two fresh-target continuation cycles beside a frozen control. The runner now calls `SASRecLearner.train`, `begin_continuation`, `score` and `fork`. Temporary artifacts are removed on leaving this cell; use the CLI with a new output directory to retain artifacts.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from cp4285.cli import demo

with TemporaryDirectory() as temporary:
    result = demo(Path(temporary) / 'synthetic')
assert result['synthetic']
assert result['checkpoint_reload_max_difference'] == 0
assert all(p['frozen'] == result['retention'][0]['frozen'] for p in result['retention'])
result['learner'], result['retention']

## Incremental PureSVD: same methods, different learning unit
Each supplied chunk contains new complete user rows in a fixed item catalogue. This is **not** streamed MarkovQRSVD and not an equivalent exposure budget to SASRec targets. The actual classical stream uses the same methods below; batch sweeps retain explicit refits.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from cp4285.models.classical import IncrementalQRSVD

rows = csr_matrix(np.random.default_rng(4285).random((24, 8), dtype=np.float32))
learner = IncrementalQRSVD(k=3, residual_rank=2, seed=4285)
learner.train(rows[:12])
before = learner.score(rows[:2])
branch = learner.fork()
branch.begin_continuation()
branch.train(rows[12:])
np.testing.assert_array_equal(learner.score(rows[:2]), before)
assert branch.score(rows[:2], candidates=(7, 0, 2)).shape == (2, 3)
branch.describe()

## Before comparison
Both evaluators belong to `experiment`, but retain different splits, seen-item rules, candidate catalogues and tie policies. Do not compare these software examples as research results. Agree task/horizon, original identities, split/cohort, vocabulary, candidates, metrics, exposure units, model capacity, schedule and seeds first. Equal frozen metrics alone do not prove state isolation; `tests/test_learners.py` checks weights, optimizer bindings/state and RNG. CPU checks do not certify CUDA/MPS, thread-safe training or restart support.